Q1
(a) Use total least squares to fit the lines to only using the data corresponding to the first line. Report the resulting parameters.

In [2]:
import numpy as np
import cv2 as cv  # convention

# Load dataset
D = np.genfromtxt("lines.csv", delimiter=",", skip_header=1)
x1, y1 = D[:, 0], D[:, 3]

# Stack points
X = np.vstack((x1, y1)).T

# Center data
X_mean = X.mean(axis=0)
X_centered = X - X_mean

# Perform SVD
U, S, Vt = np.linalg.svd(X_centered)

# Normal vector to the line (smallest singular value)
a, b = Vt[-1]

# Compute c so line passes through centroid
c = -(a * X_mean[0] + b * X_mean[1])

print(f"Total Least Squares Line: {a:.4f} x + {b:.4f} y + {c:.4f} = 0")


Total Least Squares Line: 0.7736 x + -0.6337 y + -3.7942 = 0


(b) use all the points as indicate in the code snippet below and fit three lines. 
Hint:
Run RANSAC to find a line, mask the consensus and run again and so on to find the three lines

In [3]:
from sklearn.linear_model import RANSACRegressor

X_all = D[:, :3].flatten()
Y_all = D[:, 3:].flatten()

points = np.vstack((X_all, Y_all)).T

lines = []
remaining_points = points.copy()

for i in range(3):
    model = RANSACRegressor(min_samples=2, residual_threshold=1.0)
    model.fit(remaining_points[:, 0].reshape(-1, 1), remaining_points[:, 1])
    
    slope = model.estimator_.coef_[0]
    intercept = model.estimator_.intercept_
    lines.append((slope, intercept))
    
    # Mask inliers
    inlier_mask = model.inlier_mask_
    remaining_points = remaining_points[~inlier_mask]

for idx, (m, c) in enumerate(lines, 1):
    print(f"Line {idx}: y = {m:.4f} x + {c:.4f}")


Line 1: y = -0.4595 x + 1.9176
Line 2: y = 1.0496 x + 0.9619
Line 3: y = 1.2071 x + -6.1759
